# Exploratory data analysis
Exploration only. All reusable logic lives in `src/`. Run after `python -m src.data.prepare`. Figures are written to `docs/images/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.config import load_params, resolve_path

params = load_params()
df = pd.read_parquet(resolve_path(params["data"]["processed_path"]))
figdir = Path("docs/images")
figdir.mkdir(parents=True, exist_ok=True)
df.head()

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
df["sales"].describe()

## Trend: monthly mean sales across all series

In [ ]:
monthly = df.set_index("date").resample("MS")["sales"].mean()
ax = monthly.plot(figsize=(12, 4), title="Monthly mean daily sales")
ax.set_ylabel("mean sales")
ax.figure.tight_layout()
ax.figure.savefig(figdir / "trend.png")
plt.show()

## Weekly seasonality

In [ ]:
dow = df.groupby(df["date"].dt.day_of_week)["sales"].mean()
ax = dow.plot.bar(figsize=(8, 4), title="Mean sales by day of week (0=Mon)")
ax.set_ylabel("mean sales")
ax.figure.tight_layout()
ax.figure.savefig(figdir / "weekly.png")
plt.show()

## Yearly seasonality and distribution

In [ ]:
by_month = df.groupby(df["date"].dt.month)["sales"].mean()
ax = by_month.plot(figsize=(10, 4), title="Mean sales by month")
ax.set_ylabel("mean sales")
ax.set_xlabel("month")
ax.figure.tight_layout()
ax.figure.savefig(figdir / "yearly.png")
plt.show()
ax = df.boxplot(column="sales", by="store_id", figsize=(10, 4))
ax.set_title("Sales distribution by store")
ax.set_xlabel("store_id")
ax.figure.savefig(figdir / "store_distribution.png")
plt.show()

## Findings (copied into docs/DECISIONS.md)
- The panel is complete after cleaning: one row per store/item/day.
- Weekly seasonality is strong: weekends differ from weekdays.
- Yearly seasonality and holidays are visible in monthly means.
- Store and item levels differ substantially, so identifiers are useful features.
- A handful of extreme values were winsorised during cleaning.